# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR² clinical oncology dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and referencing all data entities by their `@id` fields.

### Dataset Source
The dataset source is provided via [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) URL. All dataset entities (record sets, fields, columns) are referenced by their `@id` per best practice.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant` for inspection.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Discover available record sets, their associated `@id` fields, and the fields (columns) contained in each record set. This is critical for referencing them programmatically throughout the workflow.

In [ ]:
# List all record sets with their @id and human name
record_set_objs = dataset.metadata.record_sets

if not record_set_objs:
    print("No record sets defined in the Croissant metadata!")
else:
    print("Available record sets:")
    for rs in record_set_objs:
        print(f"@id: {rs['@id']} | name: {rs['name']}")

In [ ]:
# For each record set, print its available fields, by field @id and name
record_set_to_fields = {}
for rs in dataset.metadata.record_sets:
    print(f"\nRecord set: {rs['name']} (@id: {rs['@id']})")
    if 'fields' in rs and rs['fields']:
        for f in rs['fields']:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
            field_name = f.get('name', None) if isinstance(f, dict) else None
            print(f"  Field @id: {field_id}" + (f" | name: {field_name}" if field_name else ""))
        record_set_to_fields[rs['@id']] = [f['@id'] if isinstance(f, dict) and '@id' in f else str(f) for f in rs['fields']]
    else:
        print("  No fields available.")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames. All references are by `@id` for both record set and fields.

In [ ]:
# Collect all record set @ids to extract
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    # Use the mlcroissant API to extract records, which are dictionaries with field @id as keys
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set @id: {record_set_id}")
    else:
        print(f"No records found in record set @id: {record_set_id}")

if dataframes:
    # Pick the first record set as the default for analysis
    primary_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for primary record set (@id: {primary_record_set_id}):\n", dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records based on numeric field thresholds, normalize, and optionally group by categorical fields.

> All field and record set references use their `@id`.

In [ ]:
import numpy as np

# Identify a suitable numeric field by @id
df = dataframes.get(primary_record_set_id)
numeric_field_id = None

# Try to find a numeric field in the dataframe columns
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field available in the dataset for EDA.")
else:
    threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization (z-score)
    col_z = f"{numeric_field_id}_normalized"
    filtered_df[col_z] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_z]].head())

    # Try to find a categorical/group field
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No categorical field for grouping found in record set.")

## 5. Visualization
Visualize numeric field distribution and relationship to categorical/grouped fields (if possible).

> If the field names/IDs are not human-readable, refer to the overview above for mapping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and primary_record_set_id in dataframes:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- This notebook demonstrated dataset loading, inspection, referencing, and EDA with `mlcroissant` on a clinical oncology dataset.
- All manipulations referenced dataset entities via their canonical `@id` in both code and logic, ensuring reproducibility and clarity.
- The methods here can be adapted for any Croissant-conformant dataset, using the introspection patterns above.

<span style="color:gray">Notebook authored for FAIR²/Clinicopathological colorectal cancer example by AI.</span>